# 03 - Hierarchical Clustering

**Purpose**: Interactive hierarchical clustering with dendrograms

**Features**:
- Dendrogram visualization
- Different linkage methods (ward, complete, average)
- Optimal cut-off selection
- Cluster comparison with K-Means

---

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score, adjusted_rand_score

from notebook_utils import (
    setup_notebook, plot_cluster_distribution, plot_cluster_profiles
)

%matplotlib inline
print("✓ Imports loaded")

In [ ]:
# Load state
cfg, state = setup_notebook("Hierarchical Clustering", "germany")

df_latest = state.load('df_latest')
feature_cols = state.load('feature_cols')
kmeans_labels = state.load('kmeans_labels', default=None)
kmeans_scaler = state.load('kmeans_scaler')

# Use same features as K-Means for comparison
X_scaled = kmeans_scaler.transform(df_latest[feature_cols].fillna(df_latest[feature_cols].median()))

## Configuration

In [ ]:
# Configuration
LINKAGE_METHOD = 'ward'  # Options: 'ward', 'complete', 'average', 'single'
N_CLUSTERS = 5
DENDROGRAM_SAMPLE = 500  # Number of samples for dendrogram (full dataset can be slow)

print(f"Configuration: Linkage={LINKAGE_METHOD}, Clusters={N_CLUSTERS}")

## Dendrogram Visualization

In [ ]:
# Sample data for dendrogram
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), min(DENDROGRAM_SAMPLE, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

# Calculate linkage
Z = linkage(X_sample, method=LINKAGE_METHOD)

# Plot dendrogram
plt.figure(figsize=(15, 8))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_font_size=10, 
          show_leaf_counts=True)
plt.title(f'Hierarchical Clustering Dendrogram ({LINKAGE_METHOD.capitalize()} Linkage)', 
         fontsize=14, fontweight='bold')
plt.xlabel('Cluster Size', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.axhline(y=Z[-N_CLUSTERS, 2], c='red', linestyle='--', 
           label=f'Cut for {N_CLUSTERS} clusters')
plt.legend()
plt.tight_layout()
plt.show()

## Fit Hierarchical Clustering

In [ ]:
# Fit hierarchical clustering on full dataset
hierarchical = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage=LINKAGE_METHOD)
hier_labels = hierarchical.fit_predict(X_scaled)

# Calculate metrics
hier_silhouette = silhouette_score(X_scaled, hier_labels)

print(f"\n✓ Hierarchical clustering complete")
print(f"  Silhouette Score: {hier_silhouette:.3f}")

# Add to dataframe
df_result = df_latest.copy()
df_result['cluster_hier'] = hier_labels

## Cluster Analysis

In [ ]:
# Cluster distribution
cluster_counts = plot_cluster_distribution(df_result, 'cluster_hier', 
                                          f'Hierarchical Clustering ({LINKAGE_METHOD})')

In [ ]:
# Cluster profiles
hier_profiles = df_result.groupby('cluster_hier')[feature_cols].mean()
plot_cluster_profiles(hier_profiles, f'Hierarchical Cluster Profiles ({LINKAGE_METHOD})')

## Comparison with K-Means

In [ ]:
if kmeans_labels is not None:
    # Calculate ARI (Adjusted Rand Index)
    ari = adjusted_rand_score(kmeans_labels, hier_labels)
    
    print(f"\n📊 Comparison with K-Means:")
    print(f"  Adjusted Rand Index: {ari:.3f}")
    print(f"  Interpretation: {ari:.1%} agreement between clusterings")
    
    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(kmeans_labels, hier_labels)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.xlabel('Hierarchical Cluster', fontsize=12)
    plt.ylabel('K-Means Cluster', fontsize=12)
    plt.title('Confusion Matrix: K-Means vs Hierarchical', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️  K-Means results not found. Run 02_KMeans_Clustering.ipynb first for comparison.")

## Save Results

In [ ]:
state.save('hierarchical_labels', hier_labels)
state.save('hierarchical_results', df_result)
state.save('hierarchical_metrics', {
    'n_clusters': N_CLUSTERS,
    'linkage': LINKAGE_METHOD,
    'silhouette': hier_silhouette
})

print("\n✓ Results saved to state")
print("\n📝 Next: 04_DBSCAN_Clustering.ipynb or 05_Algorithm_Comparison.ipynb")